# Normalización completa del dataset NASA de Galicia

## 1. Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from difflib import get_close_matches

## 2. Cargar el dataset NASA a normalizar

In [2]:
# Definir la ruta del dataset NASA
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\05 - Nasa\01 - nasa municipios normalizados.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

Filas cargadas: 755178
Columnas disponibles: ['fecha', 'codigo_municipio', 'latitud', 'longitud', 'radiacion_solar', 'temperatura_humedad', 'presion_superficie', 'humedad_especifica', 'municipio']
Tamaño del dataset: 755178 filas x 9 columnas


,fecha,codigo_municipio,latitud,longitud,radiacion_solar,temperatura_humedad,presion_superficie,humedad_especifica,municipio
0,2000-01-01,15001,43.208955,-8.294335,3.75,1.93,97.56,3.98,abegondo
1,2000-01-01,15002,42.890631,-8.645921,7.97,5.15,100.18,4.94,ames
2,2000-01-01,15003,43.222655,-8.012332,3.75,1.93,97.56,3.98,aranga
3,2000-01-01,15004,43.437931,-8.257343,3.75,5.65,100.50,5.02,ares
4,2000-01-01,15006,42.926506,-8.182868,7.97,1.93,97.56,3.98,arzúa


In [3]:
# Normalizar la columna de fecha a tipo datetime y dejar solo la fecha (sin hora)
# Ajusta el nombre de la columna si no es exactamente 'fecha'
col_fecha = 'fecha' if 'fecha' in df.columns else df.columns[0]  # Asume que la primera columna es la fecha si no existe 'fecha'
df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce').dt.date
print(f"Columna '{col_fecha}' normalizada. Ejemplo de valores:")
print(df[col_fecha].dropna().astype(str).unique()[:5])

Columna 'fecha' normalizada. Ejemplo de valores:
['2000-01-01' '2000-01-02' '2000-01-03' '2000-01-04' '2000-01-05']


In [4]:
# Filtrar registros solo entre el 1 de enero de 2000 y el 31 de diciembre de 2022
fecha_inicio = pd.to_datetime('2000-01-01').date()
fecha_fin = pd.to_datetime('2022-12-31').date()
df = df[(df[col_fecha] >= fecha_inicio) & (df[col_fecha] <= fecha_fin)]
print(f'Registros tras filtrar por fecha: {len(df)}')

Registros tras filtrar por fecha: 755089


In [5]:
# Guardar el dataset limpio en la ruta indicada
import os
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\05 - Nasa'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - nasa normalizado completo.csv')
df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset limpio guardado en: {archivo_export}')

Dataset limpio guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\05 - Nasa\01 - nasa normalizado completo.csv


### Normalización de tipos tras la exportación del dataset de NASA
Ejecuta la siguiente celda para revisar y normalizar los tipos de datos del archivo final exportado.

In [6]:
# Cargar y normalizar tipos del dataset final de NASA
import pandas as pd

archivo_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\05 - Nasa\01 - nasa normalizado completo.csv'

# Cargar con low_memory=False para evitar warnings
df_check = pd.read_csv(archivo_export, low_memory=False)

# Convertir fecha
if 'fecha' in df_check.columns:
    df_check['fecha'] = pd.to_datetime(df_check['fecha'], errors='coerce').dt.date

# Convertir a numérico las columnas que deberían serlo (ajusta la lista según tus necesidades)
cols_numericas = [
    'lst_day', 'lst_night', 'ndvi', 'evi', 'precipitacion', 'pet', 'vpd', 'et', 'lai', 'fpar'
 ]
for col in cols_numericas:
    if col in df_check.columns:
        df_check[col] = pd.to_numeric(df_check[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

# Revisar tipos finales
print('Tipos de datos tras normalización:')
print(df_check.dtypes)
df_check.head()

Tipos de datos tras normalización:
fecha                   object
codigo_municipio         int64
latitud                float64
longitud               float64
radiacion_solar        float64
temperatura_humedad    float64
presion_superficie     float64
humedad_especifica     float64
municipio               object
dtype: object


,fecha,codigo_municipio,latitud,longitud,radiacion_solar,temperatura_humedad,presion_superficie,humedad_especifica,municipio
0,2000-01-01,15001,43.208955,-8.294335,3.75,1.93,97.56,3.98,abegondo
1,2000-01-01,15002,42.890631,-8.645921,7.97,5.15,100.18,4.94,ames
2,2000-01-01,15003,43.222655,-8.012332,3.75,1.93,97.56,3.98,aranga
3,2000-01-01,15004,43.437931,-8.257343,3.75,5.65,100.50,5.02,ares
4,2000-01-01,15006,42.926506,-8.182868,7.97,1.93,97.56,3.98,arzúa


In [7]:
# Guardar el DataFrame normalizado de NASA con tipos corregidos en un nuevo archivo
ruta_export_final = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\05 - Nasa'
archivo_export_final = ruta_export_final + '\\02 - nasa normalizado completo final.csv'
df_check.to_csv(archivo_export_final, index=False, encoding='utf-8')
print(f'Dataset final normalizado guardado en: {archivo_export_final}')

Dataset final normalizado guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\05 - Nasa\02 - nasa normalizado completo final.csv
